In [1]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

Using device: cuda


# AI Privacy Module 11: Membership Inference Attack - Hands-On Labs

## Course Overview
This notebook implements membership inference attacks (MIA) step-by-step:
- **Section 4**: Implementing the Target Model
- **Section 5**: Training Shadow Models
- **Section 6**: Building the Attack Classifier
- **Section 7**: Executing the Attack
- **Section 8**: Analyzing Results & Defense Mechanisms

Each section includes code, explanations, and exercises.

## Section 4: Implementing the Target Model

### Overview
The target model is the model we want to attack. We'll:
1. Create a simple neural network classifier
2. Split data into member and non-member datasets
3. Train the target model on the member dataset
4. Evaluate its performance

### Key Concepts
- **Member Data**: Samples used to train the target model
- **Non-Member Data**: Samples NOT used to train the target model
- The attack will try to determine whether a given sample is a member or not

In [2]:
import subprocess
import sys

# Install the library
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "git+https://github.com/PandaSt0rm/htb-ai-library"])

0

In [3]:
# Step 1: Standard imports
import os
import json
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [4]:
# Step 2: Import HTB AI Library components
from htb_ai_library import (
    set_reproducibility, use_htb_style,
    MLP, AttackModel,
    load_adult_census,
    train_fixed_epochs, train_with_early_stopping, evaluate_model,
    get_model_predictions, prepare_attack_data, create_dataloader,
    plot_training_history, plot_overfitting_gap, plot_confidence_distributions,
    plot_shadow_confidence_distributions, plot_attack_roc_curve, plot_precision_recall_curve,
    plot_attack_accuracy_comparison, analyze_attack_decision_boundary, plot_decision_boundary,
)

In [5]:
# Step 3: Configure execution environment
RANDOM_SEED = 1337
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_reproducibility(RANDOM_SEED)
use_htb_style()

print(f"Random Seed: {RANDOM_SEED}")
print(f"Device: {DEVICE}")

Random Seed: 1337
Device: cuda


In [6]:
# Step 4: Setup output directories
OUTPUT_DIR = "output"
MODEL_DIR = f"{OUTPUT_DIR}/models"
FIGS_DIR = "figs"
FIG_PREFIX = "Introduction_"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

DATASET_CONFIG = {
    "num_classes": 2,
}

print(f"Output directories created:")
print(f"  Models: {MODEL_DIR}")
print(f"  Figures: {FIGS_DIR}")

Output directories created:
  Models: output/models
  Figures: figs


In [7]:
# Step 5: Define model configurations
# Target model: Intentionally configured to overfit
TARGET_MODEL_CONFIG = {
    "hidden_layers": [256, 128],
    "dropout": 0.0,  # No dropout to maximize overfitting
    "epochs": 100,
    "batch_size": 32,
    "learning_rate": 0.001,
}

# Shadow models: Similar overfitting but faster training
SHADOW_MODEL_CONFIG = {
    "num_shadow_models": 5,
    "hidden_layers": [128, 64],
    "dropout": 0.3,
    "epochs": 100,
    "batch_size": 64,
    "learning_rate": 0.001,
    "early_stopping_patience": 10,
    "shadow_data_size": 0.5,
}

# Attack model: Simple classifier for membership inference
ATTACK_MODEL_CONFIG = {
    "hidden_layers": [64, 32],
    "dropout": 0.2,
    "epochs": 100,
    "batch_size": 128,
    "learning_rate": 0.001,
    "early_stopping_patience": 15,
}

print("Model configurations defined")
print(f"\nTarget Model: {TARGET_MODEL_CONFIG['hidden_layers']} (dropout={TARGET_MODEL_CONFIG['dropout']})")
print(f"Shadow Models: {SHADOW_MODEL_CONFIG['num_shadow_models']} models with {SHADOW_MODEL_CONFIG['hidden_layers']} (dropout={SHADOW_MODEL_CONFIG['dropout']})")
print(f"Attack Model: {ATTACK_MODEL_CONFIG['hidden_layers']} (dropout={ATTACK_MODEL_CONFIG['dropout']})")

Model configurations defined

Target Model: [256, 128] (dropout=0.0)
Shadow Models: 5 models with [128, 64] (dropout=0.3)
Attack Model: [64, 32] (dropout=0.2)


### Phase 2: Load Data

Load the Adult Census dataset which is split into three disjoint subsets:
- **Target Training Data** (members) - Data used to train the target model
- **Shadow Training Data** - Data for training shadow models
- **Attack Evaluation Data** (non-members) - Data to test the attack

In [8]:
# Load Adult Census dataset
print("Loading Adult Census dataset...")
X_target, y_target, X_shadow, y_shadow, X_attack_eval, y_attack_eval, num_features = load_adult_census(
    random_state=RANDOM_SEED
)

print(f"\nDataset loaded: {num_features} features")
print(f"  Target training (members): {len(X_target)} samples")
print(f"  Shadow training: {len(X_shadow)} samples")
print(f"  Attack evaluation (non-members): {len(X_attack_eval)} samples")
print(f"\nTotal samples: {len(X_target) + len(X_shadow) + len(X_attack_eval)}")

Loading Adult Census dataset...

Dataset loaded: 14 features
  Target training (members): 24421 samples
  Shadow training: 12210 samples
  Attack evaluation (non-members): 12211 samples

Total samples: 48842


### Phase 3: Train Target Model

Train the target model with intentional overfitting to create a vulnerability for the attack:
1. Zero dropout (no regularization)
2. No early stopping (train full 100 epochs)
3. This maximizes the gap between training and test accuracy

In [9]:
# Prepare data for target model training
print("\n" + "=" * 60)
print("Training Target Model")
print("=" * 60)

# Normalize using StandardScaler
scaler = StandardScaler()
X_target_norm = scaler.fit_transform(X_target)
X_attack_eval_norm = scaler.transform(X_attack_eval)

print(f"Data normalized")
print(f"  Target shape: {X_target_norm.shape}")
print(f"  Attack eval shape: {X_attack_eval_norm.shape}")


Training Target Model
Data normalized
  Target shape: (24421, 14)
  Attack eval shape: (12211, 14)


In [10]:
# Create DataLoaders (no validation split - we want full overfitting)
train_loader = create_dataloader(
    X_target_norm, y_target, 
    TARGET_MODEL_CONFIG['batch_size']
)
test_loader = create_dataloader(
    X_attack_eval_norm, y_attack_eval,
    TARGET_MODEL_CONFIG['batch_size'], 
    shuffle=False
)

print(f"DataLoaders created")
print(f"  Train batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

DataLoaders created
  Train batches: 764
  Test batches: 382


In [11]:
# Initialize target model
target_model = MLP(
    input_size=num_features,
    hidden_layers=TARGET_MODEL_CONFIG['hidden_layers'],
    num_classes=DATASET_CONFIG['num_classes'],
    dropout=TARGET_MODEL_CONFIG['dropout']
)

print(f"\nTarget Model Architecture:")
print(f"  Input: {num_features} features")
print(f"  Hidden: {TARGET_MODEL_CONFIG['hidden_layers']}")
print(f"  Output: {DATASET_CONFIG['num_classes']} classes")
print(f"  Dropout: {TARGET_MODEL_CONFIG['dropout']} (ZERO - maximizes overfitting)")
print(f"  Training epochs: {TARGET_MODEL_CONFIG['epochs']} (no early stopping)")


Target Model Architecture:
  Input: 14 features
  Hidden: [256, 128]
  Output: 2 classes
  Dropout: 0.0 (ZERO - maximizes overfitting)
  Training epochs: 100 (no early stopping)


In [12]:
# Train target model with fixed epochs (intentional overfitting)
history = train_fixed_epochs(
    target_model, train_loader, test_loader,
    device=DEVICE,
    epochs=TARGET_MODEL_CONFIG['epochs'],
    learning_rate=TARGET_MODEL_CONFIG['learning_rate']
)

print("\nTarget model training complete!")

Training: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [04:03<00:00,  2.43s/it]


Target model training complete!


In [13]:
# Evaluate target model - measure the overfitting gap
train_acc, _, _ = evaluate_model(target_model, train_loader, DEVICE)
test_acc, _, _ = evaluate_model(target_model, test_loader, DEVICE)

overfitting_gap = train_acc - test_acc

print(f"\nTarget Model Performance:")
print(f"  Training Accuracy (members): {train_acc:.4f}")
print(f"  Test Accuracy (non-members): {test_acc:.4f}")
print(f"  Overfitting Gap:             {overfitting_gap:.4f}")
print(f"\n💡 This {overfitting_gap*100:.2f}% gap is the vulnerability the attack will exploit!")


Target Model Performance:
  Training Accuracy (members): 0.9400
  Test Accuracy (non-members): 0.8238
  Overfitting Gap:             0.1162

💡 This 11.62% gap is the vulnerability the attack will exploit!


In [14]:
# Visualize training history and overfitting gap
fig1 = plot_training_history(
    history, 
    title="Target Model Training History",
    save_path=os.path.join(FIGS_DIR, f"{FIG_PREFIX}training_history.png")
)
fig2 = plot_overfitting_gap(
    train_acc, 
    test_acc, 
    save_path=os.path.join(FIGS_DIR, f"{FIG_PREFIX}overfitting_gap.png")
)

plt.show()
print(f"\nVisualizations saved to {FIGS_DIR}/")


Visualizations saved to figs/


## Summary: What We've Accomplished

This section (Section 4 from the module) implements the **target model** - the victim of the membership inference attack:

✅ **Setup**: Configured environment, imports, reproducibility, and output directories  
✅ **Data Loading**: Loaded Adult Census dataset with 3 disjoint splits  
✅ **Target Model Training**: Trained an intentionally overfit neural network  
✅ **Performance Analysis**: Measured the overfitting gap (training vs test accuracy)  
✅ **Visualization**: Plotted training curves and the vulnerability  

### Key Takeaways

1. **The Overfitting Gap** is the vulnerability: Members (training data) get higher confidence predictions than non-members (test data)
2. **Zero Dropout**: We removed all regularization to maximize memorization
3. **No Early Stopping**: We trained all 100 epochs to ensure the model memorizes training data
4. **The Gap = Attack Signal**: This behavioral difference is what enables membership inference

### Next Steps: Sections 5-8

The remaining sections will build on this vulnerable model:

- **Section 5**: Train shadow models to learn membership patterns
- **Section 6**: Build the attack classifier to distinguish members from non-members  
- **Section 7**: Execute the attack and measure success
- **Section 8**: Analyze results and explore defense mechanisms

When you complete these sections, provide me the content and I'll upload them to GitHub! 📊

## Section 5: Training Shadow Models

### Overview
Now that we understand the vulnerability in the target model, we train 5 shadow models to learn membership patterns:
- Train on different random subsets of shadow data
- Collect predictions from members and non-members
- Create attack training dataset with 61,050 samples
- Verify shadow models exhibit similar overfitting behavior

In [15]:
# Create shadow model data splits
print("\n" + "=" * 60)
print("Training Shadow Models")
print("=" * 60)

shadow_splits = []
for i in range(SHADOW_MODEL_CONFIG['num_shadow_models']):
    seed = RANDOM_SEED + i
    X_train_s, X_out_s, y_train_s, y_out_s = train_test_split(
        X_shadow, y_shadow, train_size=SHADOW_MODEL_CONFIG['shadow_data_size'],
        random_state=seed, stratify=y_shadow
    )
    shadow_splits.append((X_train_s, X_out_s, y_train_s, y_out_s))

print(f"\nCreated {len(shadow_splits)} shadow model data splits")
print(f"Samples per shadow model: ~{len(shadow_splits[0][0])} in, ~{len(shadow_splits[0][1])} out")


Training Shadow Models

Created 5 shadow model data splits
Samples per shadow model: ~6105 in, ~6105 out


In [16]:
# Train shadow models and collect predictions
all_attack_X = []
all_attack_y = []
all_preds_in = []
all_preds_out = []

for i, (X_train_s, X_out_s, y_train_s, y_out_s) in enumerate(shadow_splits):
    print(f"\nTraining Shadow Model {i+1}/{SHADOW_MODEL_CONFIG['num_shadow_models']}")

    # Normalize using target scaler for transferability
    X_train_s_norm = scaler.transform(X_train_s)
    X_out_s_norm = scaler.transform(X_out_s)

    # Create validation split for early stopping
    X_tr_s, X_val_s, y_tr_s, y_val_s = train_test_split(
        X_train_s_norm, y_train_s, test_size=0.2,
        random_state=RANDOM_SEED + i, stratify=y_train_s
    )
    train_loader_s = create_dataloader(X_tr_s, y_tr_s, SHADOW_MODEL_CONFIG['batch_size'])
    val_loader_s = create_dataloader(X_val_s, y_val_s, SHADOW_MODEL_CONFIG['batch_size'], shuffle=False)

    # Initialize and train shadow model
    shadow_model = MLP(
        input_size=num_features,
        hidden_layers=SHADOW_MODEL_CONFIG['hidden_layers'],
        num_classes=DATASET_CONFIG['num_classes'],
        dropout=SHADOW_MODEL_CONFIG['dropout']
    )
    train_with_early_stopping(
        shadow_model, train_loader_s, val_loader_s,
        device=DEVICE,
        epochs=SHADOW_MODEL_CONFIG['epochs'],
        learning_rate=SHADOW_MODEL_CONFIG['learning_rate'],
        patience=SHADOW_MODEL_CONFIG['early_stopping_patience'],
        verbose=False
    )

    # Collect predictions on members and non-members
    preds_in = get_model_predictions(shadow_model, X_train_s_norm, DEVICE)
    preds_out = get_model_predictions(shadow_model, X_out_s_norm, DEVICE)

    # Transform to attack features and accumulate
    attack_X_s, attack_y_s = prepare_attack_data(preds_in, preds_out, y_train_s, y_out_s)
    all_attack_X.append(attack_X_s)
    all_attack_y.append(attack_y_s)
    all_preds_in.append(preds_in)
    all_preds_out.append(preds_out)

    # Verify overfitting gap exists
    full_train_loader_s = create_dataloader(X_train_s_norm, y_train_s,
                                            SHADOW_MODEL_CONFIG['batch_size'], shuffle=False)
    full_out_loader_s = create_dataloader(X_out_s_norm, y_out_s,
                                          SHADOW_MODEL_CONFIG['batch_size'], shuffle=False)
    train_acc_s, _, _ = evaluate_model(shadow_model, full_train_loader_s, DEVICE)
    out_acc_s, _, _ = evaluate_model(shadow_model, full_out_loader_s, DEVICE)
    print(f"  Shadow {i+1} - Train Acc: {train_acc_s:.4f}, Out Acc: {out_acc_s:.4f}")

print("\n✅ Shadow model training complete!")


Training Shadow Model 1/5
  Shadow 1 - Train Acc: 0.8588, Out Acc: 0.8509

Training Shadow Model 2/5
  Shadow 2 - Train Acc: 0.8572, Out Acc: 0.8506

Training Shadow Model 3/5
  Shadow 3 - Train Acc: 0.8606, Out Acc: 0.8539

Training Shadow Model 4/5
  Shadow 4 - Train Acc: 0.8701, Out Acc: 0.8483

Training Shadow Model 5/5
  Shadow 5 - Train Acc: 0.8649, Out Acc: 0.8457

✅ Shadow model training complete!


In [17]:
# Combine attack training data from all shadow models
attack_X = np.concatenate(all_attack_X, axis=0)
attack_y = np.concatenate(all_attack_y, axis=0)

print(f"\nTotal attack training samples: {len(attack_X)}")
print(f"  Members: {np.sum(attack_y == 1)}")
print(f"  Non-members: {np.sum(attack_y == 0)}")
print(f"\nAttack feature dimensions: {attack_X.shape[1]}")
print(f"Example member feature: {attack_X[0].round(3)}")
print(f"Example non-member feature: {attack_X[len(attack_X)//2].round(3)}")


Total attack training samples: 61050
  Members: 30525
  Non-members: 30525

Attack feature dimensions: 4
Example member feature: [0.893 0.107 1.    0.   ]
Example non-member feature: [0.964 0.036 1.    0.   ]


In [18]:
# Visualize shadow model confidence distributions
plot_shadow_confidence_distributions(
    all_preds_in, all_preds_out,
    save_path=os.path.join(FIGS_DIR, f"{FIG_PREFIX}shadow_confidence.png")
)

plt.show()
print(f"\nConfidence distribution visualization saved to {FIGS_DIR}/")


Confidence distribution visualization saved to figs/


In [19]:
# Analyze attack data statistics
member_confidences = attack_X[attack_y == 1, :2].max(axis=1)
non_member_confidences = attack_X[attack_y == 0, :2].max(axis=1)

print(f"\nAttack Data Statistics:")
print(f"  Member confidence - Mean: {member_confidences.mean():.4f}, Std: {member_confidences.std():.4f}")
print(f"  Non-member confidence - Mean: {non_member_confidences.mean():.4f}, Std: {non_member_confidences.std():.4f}")
print(f"  Confidence gap: {member_confidences.mean() - non_member_confidences.mean():.4f}")
print(f"\n💡 This subtle confidence gap is the signal the attack classifier will learn to detect!")


Attack Data Statistics:
  Member confidence - Mean: 0.8569, Std: 0.1592
  Non-member confidence - Mean: 0.8571, Std: 0.1591
  Confidence gap: -0.0002

💡 This subtle confidence gap is the signal the attack classifier will learn to detect!


## Section 6: Building the Attack Classifier

### Overview
Now we train the attack model to distinguish members from non-members:
- Split attack data into train/validation/test sets
- Build small attack model (64, 32 neurons)
- Train with early stopping on shadow data
- Evaluate performance and analyze decision boundary

In [ ]:
# Split attack data into train/test
X_attack_train, X_attack_test, y_attack_train, y_attack_test = train_test_split(
    attack_X, attack_y, test_size=0.2, random_state=RANDOM_SEED, stratify=attack_y
)

print(f"\nAttack data split:")
print(f"  Training + Validation: {len(X_attack_train)} samples")
print(f"  Test: {len(X_attack_test)} samples")

# Further split training into train/validation
X_attack_tr, X_attack_val, y_attack_tr, y_attack_val = train_test_split(
    X_attack_train, y_attack_train, test_size=0.2, random_state=RANDOM_SEED, stratify=y_attack_train
)

print(f"  Training: {len(X_attack_tr)} samples")
print(f"  Validation: {len(X_attack_val)} samples")

# Create DataLoaders
attack_train_loader = create_dataloader(X_attack_tr, y_attack_tr, ATTACK_MODEL_CONFIG['batch_size'])
attack_val_loader = create_dataloader(X_attack_val, y_attack_val, ATTACK_MODEL_CONFIG['batch_size'], shuffle=False)
attack_test_loader = create_dataloader(X_attack_test, y_attack_test, ATTACK_MODEL_CONFIG['batch_size'], shuffle=False)

print(f"\nDataLoaders created with batch size {ATTACK_MODEL_CONFIG['batch_size']}")

In [ ]:
# Initialize attack model
attack_input_size = attack_X.shape[1]
attack_model = AttackModel(
    input_size=attack_input_size,
    hidden_layers=ATTACK_MODEL_CONFIG['hidden_layers'],
    dropout=ATTACK_MODEL_CONFIG['dropout']
)

print(f"\nAttack model architecture: {attack_input_size} -> {ATTACK_MODEL_CONFIG['hidden_layers']} -> 2")
print(f"Dropout: {ATTACK_MODEL_CONFIG['dropout']}")
print(f"Approximate parameters: ~2,600")

In [ ]:
# Train attack model
print("\n" + "=" * 60)
print("Training Attack Model")
print("=" * 60)

history_attack = train_with_early_stopping(
    attack_model, attack_train_loader, attack_val_loader,
    device=DEVICE,
    epochs=ATTACK_MODEL_CONFIG['epochs'],
    learning_rate=ATTACK_MODEL_CONFIG['learning_rate'],
    patience=ATTACK_MODEL_CONFIG['early_stopping_patience']
)

plot_training_history(
    history_attack,
    "Attack Model Training",
    save_path=os.path.join(FIGS_DIR, f"{FIG_PREFIX}attack_training.png")
)

plt.show()
print(f"\nAttack model training complete!")

In [ ]:
# Evaluate attack model on test set
attack_test_acc, attack_test_predictions, attack_test_probs = evaluate_model(attack_model, attack_test_loader, DEVICE)

print(f"\nAttack Model Test Performance:")
print(f"  Accuracy: {attack_test_acc:.4f}")
print(f"  Samples: {len(attack_test_predictions)}")

print("\nDetailed Classification Report:")
print(classification_report(
    y_attack_test,
    attack_test_predictions,
    target_names=['Non-Member', 'Member'],
    digits=4
))

# Save the attack model
attack_model_path = os.path.join(MODEL_DIR, "attack_model.pt")
torch.save(attack_model.state_dict(), attack_model_path)
print(f"\nAttack model saved to {attack_model_path}")

In [ ]:
# Analyze decision boundary
boundary_analysis = analyze_attack_decision_boundary(attack_model, DEVICE)

print("\nDecision Boundary Analysis:")
for cls, data in boundary_analysis.items():
    threshold_idx = np.argmin(np.abs(data['membership_probs'] - 0.5))
    threshold_conf = data['confidences'][threshold_idx]
    print(f"  Class {cls}: Membership threshold at confidence ~{threshold_conf:.3f}")

In [ ]:
# Visualize decision boundary
plot_decision_boundary(
    boundary_analysis,
    save_path=os.path.join(FIGS_DIR, f"{FIG_PREFIX}decision_boundary.png")
)

plt.show()

print(f"\n💡 Attack model has learned a confidence threshold for membership detection!")
print(f"Expected performance on target model: 65-66% accuracy (vs ~50% on shadow models)")

## Section 6: Executing and Evaluating the Attack

This section applies the trained attack model to the target model, then measures how much membership information leaks through predictions and confidence scores.

In [ ]:
# Collect target model predictions for members and non-members
print("\n" + "=" * 60)
print("Executing Membership Inference Attack")
print("=" * 60)

preds_members = get_model_predictions(target_model, X_target_norm, DEVICE)
preds_non_members = get_model_predictions(target_model, X_attack_eval_norm, DEVICE)

print(f"\nTarget model predictions collected:")
print(f"  Members: {len(preds_members)} samples")
print(f"  Non-members: {len(preds_non_members)} samples")

In [ ]:
# Prepare attack features and labels
attack_X_members, attack_y_members = prepare_attack_data(
    preds_members, np.zeros((0, preds_members.shape[1])),
    y_target, np.array([], dtype=np.int64)
)

attack_X_non_members, attack_y_non_members = prepare_attack_data(
    np.zeros((0, preds_non_members.shape[1])), preds_non_members,
    np.array([], dtype=np.int64), y_attack_eval
)

print(f"\nAttack input prepared:")
print(f"  Member features: {attack_X_members.shape}")
print(f"  Non-member features: {attack_X_non_members.shape}")

In [ ]:
# Combine attack data and run the attack
attack_X_eval = np.concatenate([attack_X_members, attack_X_non_members], axis=0)
attack_y_eval = np.concatenate([attack_y_members, attack_y_non_members], axis=0)

print(f"\nTotal attack evaluation samples: {len(attack_X_eval)}")
print(f"  Members: {np.sum(attack_y_eval == 1)}")
print(f"  Non-members: {np.sum(attack_y_eval == 0)}")

attack_eval_loader = create_dataloader(attack_X_eval, attack_y_eval, ATTACK_MODEL_CONFIG['batch_size'], shuffle=False)

_, attack_predictions, attack_probs = evaluate_model(attack_model, attack_eval_loader, DEVICE)
membership_probs = attack_probs[:, 1]

print(f"\nAttack predictions generated")
print(f"  Mean membership probability: {membership_probs.mean():.4f}")

In [ ]:
# Compute attack metrics
attack_accuracy = accuracy_score(attack_y_eval, attack_predictions)
attack_precision = precision_score(attack_y_eval, attack_predictions)
attack_recall = recall_score(attack_y_eval, attack_predictions)
attack_f1 = f1_score(attack_y_eval, attack_predictions)

print(f"\nMembership Inference Attack Results:")
print(f"  Attack Accuracy:  {attack_accuracy:.4f}")
print(f"  Attack Precision: {attack_precision:.4f}")
print(f"  Attack Recall:    {attack_recall:.4f}")
print(f"  Attack F1 Score:  {attack_f1:.4f}")
print(f"  Attack Advantage: {attack_accuracy - 0.5:.4f}")

In [ ]:
# Store results for later plots
results = {
    'attack_accuracy': attack_accuracy,
    'attack_precision': attack_precision,
    'attack_recall': attack_recall,
    'attack_f1': attack_f1,
    'attack_y_true': attack_y_eval,
    'attack_y_pred': attack_predictions,
    'attack_probs': membership_probs,
    'confidence_members': np.max(preds_members, axis=1),
    'confidence_non_members': np.max(preds_non_members, axis=1),
}

print("\nResults stored for visualization")

In [ ]:
# Generate attack visualizations
print("\n" + "=" * 60)
print("Generating Visualizations")
print("=" * 60)

auc_score = plot_attack_roc_curve(
    results['attack_y_true'],
    results['attack_probs'],
    save_path=os.path.join(FIGS_DIR, f"{FIG_PREFIX}attack_roc.png")
)
results['attack_auc'] = auc_score

plot_precision_recall_curve(
    results['attack_y_true'],
    results['attack_probs'],
    save_path=os.path.join(FIGS_DIR, f"{FIG_PREFIX}attack_pr.png")
)

plot_confidence_distributions(
    results['confidence_members'],
    results['confidence_non_members'],
    save_path=os.path.join(FIGS_DIR, f"{FIG_PREFIX}confidence_distributions.png")
)

plot_attack_accuracy_comparison(
    results,
    save_path=os.path.join(FIGS_DIR, f"{FIG_PREFIX}attack_metrics.png")
)

print(f"Attack AUC: {auc_score:.4f}")
print(f"Mean confidence - Members: {np.mean(results['confidence_members']):.4f}")
print(f"Mean confidence - Non-Members: {np.mean(results['confidence_non_members']):.4f}")

In [ ]:
# Save attack results
output = {
    'target_model': {
        'train_accuracy': float(train_acc),
        'test_accuracy': float(test_acc),
        'overfitting_gap': float(train_acc - test_acc),
    },
    'attack_results': {
        'accuracy': float(results['attack_accuracy']),
        'precision': float(results['attack_precision']),
        'recall': float(results['attack_recall']),
        'f1_score': float(results['attack_f1']),
        'auc': float(results['attack_auc']),
        'advantage': float(results['attack_accuracy'] - 0.5),
    },
    'configuration': {
        'random_seed': RANDOM_SEED,
        'num_shadow_models': SHADOW_MODEL_CONFIG['num_shadow_models'],
        'target_architecture': TARGET_MODEL_CONFIG['hidden_layers'],
        'attack_architecture': ATTACK_MODEL_CONFIG['hidden_layers'],
    }
}

results_path = os.path.join(FIGS_DIR, f"{FIG_PREFIX}attack_results.json")
with open(results_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f"\nResults saved to {results_path}")